In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm, trange

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchvision.transforms import ToTensor
from torchvision import models
from torch.utils.data import WeightedRandomSampler

np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# data augmentation pipeline
import torchvision.transforms as transforms
from torchvision import datasets

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    #transforms.RandomVerticalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_set = datasets.ImageFolder(root = 'Splitted2D/train', transform = train_transform)
val_set = datasets.ImageFolder(root = 'Splitted2D/val', transform = val_transform)
test_set = datasets.ImageFolder(root = 'Splitted2D/test', transform = test_transform)

In [ ]:
train_set.imgs

In [ ]:
idx2class = {v: k for k, v in train_set.class_to_idx.items()}

def get_class_distribution(dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for element in dataset_obj:
        y_lbl = element[1]
        y_lbl = idx2class[y_lbl]
        count_dict[y_lbl] += 1
            
    return count_dict
print("Distribution of classes: \n", get_class_distribution(train_set))

In [ ]:
BATCH_SIZE = 32

def get_class_distribution_loaders(dataloader_obj, dataset_obj):
    count_dict = {k:0 for k,v in dataset_obj.class_to_idx.items()}
    
    for _,j in dataloader_obj:
        y_idx = j.item()
        y_lbl = idx2class[y_idx]
        count_dict[str(y_lbl)] += 1
            
    return count_dict

target_list = torch.tensor(train_set.targets)
class_count = [i for i in get_class_distribution(train_set).values()]
class_weights = 1./torch.tensor(class_count, dtype=torch.float) 
print(class_weights)
###################### OUTPUT ######################
#tensor([0.0068, 0.0054, 0.0114, 0.0123])

class_weights_all = class_weights[target_list]

weighted_sampler = WeightedRandomSampler(
    weights=class_weights_all,
    num_samples=len(class_weights_all),
    replacement=False
)

train_loader = DataLoader(dataset = train_set, shuffle = False, batch_size = BATCH_SIZE, sampler = weighted_sampler)
val_loader = DataLoader(dataset = val_set, shuffle = True, batch_size = BATCH_SIZE)
test_loader = DataLoader(dataset = test_set, shuffle = True, batch_size = BATCH_SIZE)

In [ ]:
for batch in train_loader:
    x, y = batch
    print(x.shape)
    plt.figure()
    plt.imshow(x[2][0])
    plt.show()

In [ ]:
vgg16 = models.vgg16_bn(weights = 'IMAGENET1K_V1')
#vgg16.load_state_dict(torch.load("../input/vgg16bn/vgg16_bn.pth"))
print(vgg16.classifier[6].out_features) # 1000 


# Freeze training for all layers
for param in vgg16.features.parameters():
    param.require_grad = False

# Newly created modules have require_grad=True by default
num_features = vgg16.classifier[6].in_features
features = list(vgg16.classifier.children())[:-1] # Remove last layer
features.extend([nn.Linear(num_features, 4)]) # Add our layer with 4 outputs
vgg16.classifier = nn.Sequential(*features) # Replace the model classifier
print(vgg16)

In [ ]:
from torchsummary import summary
summary(vgg16, (3, 224, 224))

In [ ]:
N_EPOCHS = 75
LR = 0.01
device = torch.device('mps')
vgg16.to(device)
optimizer = Adam(vgg16.parameters(), lr = LR)
criterion = CrossEntropyLoss()


for epoch in trange(N_EPOCHS, desc = "TRAINING"):
    correct, total = 0, 0
    train_loss = 0.0
    
    train_correct, train_total = 0, 0
    for batch in tqdm(train_loader, desc = f"EPOCH {epoch + 1} IN TRAINING", leave = False):
        x, y = batch
        #print(x.shape)
        x, y = x.to(device), y.to(device)
        y_hat = vgg16(x)
        
        loss = criterion(y_hat, y)
        
        train_loss += loss.detach().cpu().item() / len(train_loader)
        train_correct += torch.sum(torch.argmax(y_hat, dim = 1) == y).detach().cpu().item()
        train_total += len(x)
        
        optimizer.zero_grad()
        #count = 0
#         for name, param in vgg16.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                 print(gradient_norm)
                #count += 1
        
        loss.backward()
        optimizer.step()
    print(f"Accuracy: {train_correct / train_total * 100:.2f}%")
        #print(torch.argmax(y_hat, dim = 1))
    for batch in tqdm(val_loader, desc = 'VALIDATION'):
        x, y = batch
        x, y = x.to(device), y.to(device)
        
        y_hat = vgg16(x)
        
        correct += torch.sum(torch.argmax(y_hat, dim = 1) == y).detach().cpu().item()
        total += len(x)
    
    print(f"EPOCH {epoch + 1} / {N_EPOCHS} loss: {train_loss:.2f} accuracy: {train_correct / train_total * 100:.2f}%")
    print(f"Validation accuracy: {correct / total * 100:.2f}%")

with torch.no_grad():
    correct, total = 0, 0
    test_loss = 0.0
    for batch in tqdm(test_loader, desc = "TESTING"):
        x, y = batch

        x, y = x.to(device), y.to(device)
        
        y_hat = vgg16(x.float())

        loss = criterion(y_hat, y)

        test_loss += loss.detach().cpu().item() / len(test_loader)

        correct += torch.sum(torch.argmax(y_hat, dim=1) == y).detach().cpu().item()
        total += len(x)
    print(f"Test loss: {test_loss:.2f}")
    print(f"Test accuracy: {correct / total * 100:.2f}%")
    

In [ ]:
x_train.shape